# 02 · NAKSHA-Footprint-UNet (deep learning, GPU)

Trains a **U-Net with a ResNet-34 encoder** that draws building outlines from aerial / drone imagery (ORI). It is exported to ONNX so the NAKSHA worker could run it on CPU.

| Model | Type | Solves (PS 26013) |
|---|---|---|
| NAKSHA-Footprint-UNet | Deep learning: semantic segmentation (GeoAI / computer vision) | AI-generated feature extraction from drone imagery / ORI |

**Runtime:** GPU. On Kaggle: *Settings → Accelerator → GPU T4 x2 / P100*, and Internet on. On Colab: *Runtime → Change runtime type → T4 GPU*.

**Data**
1. **Pre-training: Inria Aerial Image Labeling** (0.3 m, 180 images with building masks).
   - Kaggle: *Add Input* → search "Inria Aerial Image Labeling" and attach it.
   - Colab: download it from project.inria.fr/aerialimagelabeling (free registration) into Drive.
2. **Fine-tuning on GVMC (optional):** drone ORI GeoTIFFs of GVMC wards, attached as a dataset or put in Drive. The labels are `gvmc_buildings.geojson` from notebook 01 (all 98 wards).

> **Limit:** no open imagery of GVMC exists at drone resolution; satellite imagery is 10 m. Without the department's ORI, the model is trained on Inria only. It then works on similar 0.3 m imagery, but its accuracy on Visakhapatnam is unmeasured.

`QUICK = True` runs the whole chain in minutes on fake data (a smoke test). Set it to `False` for real training.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
BRANCH = "main"
QUICK = True               # True: 1-epoch smoke test on generated data; False: real training
INRIA_DIR = None           # None = search attached inputs / Drive for Inria (train/images + train/gt)
INRIA_IMAGES = 60          # how many Inria images to tile (each ≈ 90 tiles at 0.3 m)
GVMC_ORI_DIR = None        # folder of GVMC ORI GeoTIFFs for fine-tuning (optional)
GVMC_LABELS = None         # gvmc_buildings.geojson from notebook 01 (needed with GVMC_ORI_DIR)
EPOCHS_PRETRAIN = 40
EPOCHS_FINETUNE = 20
BATCH = 8

In [ ]:
# ── Environment: Kaggle, Colab or local ─────────────────────────────────────
import os, sys, subprocess, shutil, glob, json, time

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else os.path.abspath("naksha_work")
os.makedirs(WORK, exist_ok=True)
print("running on", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local", "| work dir:", WORK)


def run(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


# The code comes from the project repository (public). NAKSHA_REPO = use an existing checkout instead.
REPO_URL = "https://github.com/GVMC-NAKSHA-1/gvmc-naksha.git"
REPO = os.environ.get("NAKSHA_REPO") or os.path.join(WORK, "gvmc-naksha")
if not os.path.exists(os.path.join(REPO, ".git")) and not os.environ.get("NAKSHA_REPO"):
    run(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}")
    # Survey of India village boundaries (official GVMC outline) are stored with Git LFS.
    try:
        run("git lfs install --local && git lfs pull", cwd=REPO)
    except RuntimeError:
        print("Git LFS unavailable: wards will be clipped to the OpenStreetMap study area instead of the SoI outline.")
print("repo:", REPO)


In [ ]:
import torch
GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none")
if not GPU and not QUICK:
    raise SystemExit("No GPU. Kaggle: Settings -> Accelerator -> GPU; Colab: Runtime -> Change runtime type -> T4 GPU.")
if ON_COLAB and not QUICK:
    from google.colab import drive
    drive.mount("/content/drive")
if not os.environ.get("NAKSHA_SKIP_INSTALL"):
    run(f"{sys.executable} -m pip install -q -r ml/footprint/requirements.txt", cwd=REPO)
FP = os.path.join(REPO, "ml", "footprint")
OUTPUTS = os.path.join(WORK, "outputs")
os.makedirs(OUTPUTS, exist_ok=True)
EPOCHS_PRETRAIN, EPOCHS_FINETUNE = (1, 1) if QUICK else (EPOCHS_PRETRAIN, EPOCHS_FINETUNE)
WORKERS = 2 if (ON_KAGGLE or ON_COLAB) else 0

## 1 · Find the imagery

In [ ]:
def find_inria():
    roots = [INRIA_DIR] if INRIA_DIR else ["/kaggle/input", "/content/drive/MyDrive", "/content"]
    for root in roots:
        for imgs in glob.glob(os.path.join(root, "**", "train", "images"), recursive=True):
            gt = os.path.join(os.path.dirname(imgs), "gt")
            if os.path.isdir(gt) and glob.glob(os.path.join(imgs, "*.tif")):
                return imgs, gt
    return None, None


def fake_imagery(folder, n=2, size=2048):
    """QUICK mode without Inria: bright rectangular roofs on textured ground, with exact masks."""
    import numpy as np, rasterio
    from rasterio.transform import from_origin
    os.makedirs(os.path.join(folder, "images"), exist_ok=True)
    os.makedirs(os.path.join(folder, "gt"), exist_ok=True)
    r = np.random.default_rng(0)
    for k in range(n):
        img = r.normal(90, 12, (3, size, size)).clip(0, 255); img[1] += 25
        mask = np.zeros((size, size), np.uint8)
        for _ in range(160):
            w, h = r.integers(25, 70, 2); x, y = r.integers(0, size - 80, 2)
            img[:, y:y + h, x:x + w] = r.normal(200, 10) + r.normal(0, 5, (3, h, w)); mask[y:y + h, x:x + w] = 255
        prof = dict(driver="GTiff", height=size, width=size, crs="EPSG:32644", transform=from_origin(780000, 1965000, 0.3, 0.3))
        with rasterio.open(os.path.join(folder, "images", f"fake{k}.tif"), "w", count=3, dtype="uint8", **prof) as ds:
            ds.write(img.astype("uint8"))
        with rasterio.open(os.path.join(folder, "gt", f"fake{k}.tif"), "w", count=1, dtype="uint8", **prof) as ds:
            ds.write(mask, 1)
    return os.path.join(folder, "images"), os.path.join(folder, "gt")


images_dir, gt_dir = find_inria()
if images_dir is None:
    if not QUICK:
        raise SystemExit("Inria not found. Kaggle: Add Input -> 'Inria Aerial Image Labeling'. "
                         "Colab: put AerialImageDataset/train/{images,gt} in Drive, or set INRIA_DIR.")
    print("Inria not attached: QUICK mode uses generated imagery")
    images_dir, gt_dir = fake_imagery(os.path.join(WORK, "fake_inria"))
imgs = sorted(glob.glob(os.path.join(images_dir, "*.tif")))[: (2 if QUICK else INRIA_IMAGES)]
print(len(imgs), "pre-training images from", images_dir)

## 2 · Cut 512×512 training tiles
Tiles are cut at 0.3 m and 0.5 m because the worker sees imagery at different resolutions. The train/val/test split is by spatial block, so neighbouring tiles never leak into the test set.

In [ ]:
TILES = os.path.join(WORK, "tiles_pretrain")
shutil.rmtree(TILES, ignore_errors=True)
masks = [os.path.join(gt_dir, os.path.basename(p)) for p in imgs]
run(f"{sys.executable} prepare_data.py --image {' '.join(imgs)} --mask {' '.join(masks)} --out {TILES} "
    f"--gsd 0.3 0.5 --block {1 if QUICK else 4}", cwd=FP)

## 3 · Pre-train

In [ ]:
RUN = os.path.join(WORK, "runs", "r34")
run(f"{sys.executable} train.py --data {TILES} --out {RUN} --epochs {EPOCHS_PRETRAIN} --batch {BATCH} --workers {WORKERS}"
    + (" --encoder resnet18" if QUICK else ""), cwd=FP)
BEST = os.path.join(RUN, "best.pt")
EVAL_TILES = TILES

## 4 · Fine-tune on GVMC ORI (skipped when no ORI is given)

In [ ]:
if GVMC_ORI_DIR and GVMC_LABELS:
    ori = sorted(glob.glob(os.path.join(GVMC_ORI_DIR, "*.tif")))
    GTILES = os.path.join(WORK, "tiles_gvmc")
    run(f"{sys.executable} prepare_data.py --image {' '.join(ori)} --labels {GVMC_LABELS} --out {GTILES} --gsd 0.3 0.5", cwd=FP)
    GRUN = os.path.join(WORK, "runs", "r34_gvmc")
    run(f"{sys.executable} train.py --data {GTILES} --out {GRUN} --init {BEST} --epochs {EPOCHS_FINETUNE} "
        f"--lr 1e-4 --batch {BATCH} --workers {WORKERS}", cwd=FP)
    BEST, EVAL_TILES = os.path.join(GRUN, "best.pt"), GTILES
else:
    print("No GVMC ORI given: the model stays Inria-trained.")

## 5 · Export to ONNX (checks the worker's input/output contract) and evaluate

In [ ]:
ONNX = os.path.join(OUTPUTS, "footprint_unet_r34.onnx")
run(f"{sys.executable} export_onnx.py --ckpt {BEST} --out {ONNX}", cwd=FP)
run(f"{sys.executable} evaluate.py --onnx {ONNX} --data {EVAL_TILES} --split test --report {OUTPUTS}/eval.json", cwd=FP)
shutil.copy(os.path.join(os.path.dirname(BEST), "history.json"), OUTPUTS)
import pandas as pd
ev = json.load(open(os.path.join(OUTPUTS, "eval.json")))
display(pd.DataFrame({"NAKSHA-Footprint-UNet (deep learning)": ev["deep_learning"],
                      "classical baseline (worker fallback)": ev["classical_baseline"]}))
print("targets for GVMC: pixel IoU >= 0.70, building F1 >= 0.75")

In [ ]:
# ── Look at a few test tiles: image, label, prediction ─────────────────────
import numpy as np, cv2, onnxruntime as ort, matplotlib.pyplot as plt
sess = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
tiles = sorted(glob.glob(os.path.join(EVAL_TILES, "test", "images", "*.png")))[:3]
fig, axes = plt.subplots(len(tiles), 3, figsize=(10, 3.4 * len(tiles)), squeeze=False)
for row, p in zip(axes, tiles):
    rgb = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(p.replace(os.sep + "images" + os.sep, os.sep + "masks" + os.sep), 0)
    prob = sess.run(None, {"image": (rgb.astype(np.float32) / 255).transpose(2, 0, 1)[None]})[0][0, 0]
    for ax, im, t in zip(row, (rgb, gt, prob > 0.5), ("image", "label", "prediction")):
        ax.imshow(im, cmap=None if t == "image" else "gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
ARCHIVE_NAME = "naksha_footprint_outputs"
# ── Package the outputs ─────────────────────────────────────────────────────
archive = shutil.make_archive(os.path.join(WORK, ARCHIVE_NAME), "zip", OUTPUTS)
print("outputs:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
for f in sorted(os.listdir(OUTPUTS)):
    print("  ", f)
if ON_COLAB:
    from google.colab import files
    files.download(archive)
elif ON_KAGGLE:
    print("Kaggle: click 'Save Version' (Save & Run All) to keep /kaggle/working; download it from the Output tab.")
